# 06 - Quantum Mechanics in Action (Capstone)

This notebook brings all four postulates together by building and analyzing a complete quantum protocol: quantum teleportation.

Teleportation uses every concept we have learned:
- State space (Postulate 1): qubits as vectors
- Unitary evolution (Postulate 2): gates transform states
- Measurement (Postulate 3): collapse determines classical bits
- Composition (Postulate 4): entanglement between qubits

We will do it with raw math first, then verify with Qiskit.

In [ ]:
import numpy as np

## Setup

Alice has a qubit in an unknown state |psi> that she wants to send to Bob. They share a Bell pair (entangled qubits). Alice cannot just copy the state (no-cloning theorem). But she can teleport it.

The protocol uses 3 qubits:
- Qubit 0: Alice's state to teleport (|psi>)
- Qubit 1: Alice's half of the Bell pair
- Qubit 2: Bob's half of the Bell pair

In [ ]:
# Gates
I = np.eye(2, dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

CNOT = np.array([[1,0,0,0],
                 [0,1,0,0],
                 [0,0,0,1],
                 [0,0,1,0]], dtype=complex)

# Basis states
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# Alice's state to teleport (arbitrary unknown state)
alpha = 0.6 + 0.1j
beta = np.sqrt(1 - abs(alpha)**2) * np.exp(1j * 0.7)
psi = alpha * ket_0 + beta * ket_1
psi = psi / np.linalg.norm(psi)  # normalize

print(f"State to teleport: |psi> = {psi.round(4)}")
print(f"P(0) = {abs(psi[0])**2:.4f}, P(1) = {abs(psi[1])**2:.4f}")

## Step 1: Create the Bell Pair (Postulate 4 - Composition)

Alice and Bob share an entangled Bell pair: (|00> + |11>)/sqrt(2)

In [ ]:
# Initial 3-qubit state: |psi> tensor |00>
initial = np.kron(psi, np.kron(ket_0, ket_0))
print(f"Initial state (8D vector): {initial.round(4)}")

# Create Bell pair on qubits 1,2: apply H to qubit 1, CNOT on qubits 1,2
# H on qubit 1 only: I tensor H tensor I
H_q1 = np.kron(np.kron(I, H), I)
after_H = H_q1 @ initial

# CNOT on qubits 1,2: I tensor CNOT
CNOT_12 = np.kron(I, CNOT)
after_bell = CNOT_12 @ after_H

print(f"\nAfter Bell pair creation: {after_bell.round(4)}")
print("Qubits 1 and 2 are now entangled.")

## Step 2: Alice's Operations (Postulate 2 - Evolution)

Alice applies CNOT (control: qubit 0, target: qubit 1), then Hadamard on qubit 0.

In [ ]:
# CNOT on qubits 0,1: CNOT tensor I
CNOT_01 = np.kron(CNOT, I)
after_cnot = CNOT_01 @ after_bell

# H on qubit 0: H tensor I tensor I
H_q0 = np.kron(np.kron(H, I), I)
after_alice = H_q0 @ after_cnot

print(f"After Alice's operations: {after_alice.round(4)}")

## Step 3: Alice's Measurement (Postulate 3 - Measurement)

Alice measures her two qubits (0 and 1). She gets one of four results: 00, 01, 10, 11.

Bob's qubit (qubit 2) collapses to a state that depends on Alice's result.

In [ ]:
# Reshape to see Bob's state for each of Alice's measurement outcomes
state = after_alice.reshape(4, 2)  # 4 outcomes for Alice x 2D for Bob

print("Alice's measurement outcomes and Bob's resulting state:\n")
corrections = {
    '00': I,
    '01': X,
    '10': Z,
    '11': X @ Z,
}

labels = ['00', '01', '10', '11']
for i, label in enumerate(labels):
    bob_state = state[i]
    prob = np.linalg.norm(bob_state)**2
    bob_normalized = bob_state / np.linalg.norm(bob_state) if prob > 1e-10 else bob_state
    
    # Apply correction
    corrected = corrections[label] @ bob_normalized
    
    print(f"Alice measures |{label}>  (P={prob:.4f})")
    print(f"  Bob's raw state:  {bob_normalized.round(4)}")
    print(f"  After correction: {corrected.round(4)}")
    print(f"  Matches |psi>?    {np.allclose(np.abs(corrected), np.abs(psi))}")
    print()

## Step 4: Bob's Correction

Alice sends her 2 classical bits (measurement result) to Bob. Based on the result:

| Alice gets | Bob applies |
|-----------|-------------|
| 00 | Nothing (I) |
| 01 | X gate |
| 10 | Z gate |
| 11 | X then Z |

After correction, Bob's qubit is in state |psi>. Teleportation complete.

Important: Alice's original qubit is destroyed in the process (it collapsed during measurement). No cloning violation.

## Verify with Qiskit

In [ ]:
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector

    # Build teleportation circuit (without measurement for statevector sim)
    qc = QuantumCircuit(3)
    
    # Prepare the state to teleport on qubit 0
    qc.initialize(psi, 0)
    
    # Create Bell pair on qubits 1, 2
    qc.h(1)
    qc.cx(1, 2)
    
    # Alice's operations
    qc.cx(0, 1)
    qc.h(0)
    
    print("Teleportation circuit (before measurement):")
    print(qc.draw())
    
    sv = Statevector.from_instruction(qc)
    print(f"\nFull 3-qubit statevector: {sv.data.round(4)}")
    print(f"\nOriginal state to teleport: {psi.round(4)}")
    print("Bob's qubit will match after Alice measures and sends correction.")

except ImportError:
    print("Qiskit not installed. Our manual math verified the protocol works.")
    print("Install with: pip install qiskit")

## What We Just Did (Summary)

We implemented quantum teleportation using all four postulates:

1. **State Space**: Represented 3 qubits as an 8D vector in Hilbert space
2. **Evolution**: Applied unitary gates (H, CNOT) to transform the state
3. **Measurement**: Alice measured her qubits, collapsing the system
4. **Composition**: Used tensor products to combine qubits, created entanglement

The result: an unknown quantum state was transferred from Alice to Bob using shared entanglement and 2 classical bits of communication. No information traveled faster than light (Alice still had to send the classical bits). The original state was destroyed (no cloning violation).

This is real. This has been demonstrated experimentally. And it is one of the building blocks of quantum networks.

## Exercises

1. Teleport the state |-> instead of a random state. Verify it works.

2. What happens if Alice and Bob share a different Bell state, say (|01> + |10>)/sqrt(2), instead of (|00> + |11>)/sqrt(2)? Do the correction gates change?

3. Can you teleport an entangled qubit? Try teleporting one half of a Bell pair and check if entanglement is preserved.

In [ ]:
# Your code here
